In [1]:
!pip install yfinance psycopg2-binary pandas

In [2]:
import yfinance as yf
import pandas as pd
import psycopg2

In [3]:
symbol = "AAPL"
df = yf.download(
    symbol,
    start="2023-01-01",
    end="2025-01-01",
    interval="1d", 
    progress=False,
    threads=False
)

df = df.reset_index()
df.head()

Price,index,Close,High,Low,Open,Volume
Ticker,,AAPL,AAPL,AAPL,AAPL,AAPL
0,2023-01-03,122.982719,128.715417,122.097738,128.105769,112117500
1,2023-01-04,124.251183,126.512801,122.992546,124.772336,89113600
2,2023-01-05,122.933533,125.637638,122.677877,125.008319,80962700
3,2023-01-06,127.456772,128.115588,122.805715,123.907026,87754700
4,2023-01-09,127.977913,131.183516,127.722257,128.292580,70790800


In [6]:
df = df.rename(columns={"index": "Date"})

In [8]:
df.head()

Price,date,Close,High,Low,Open,Volume
Ticker,,AAPL,AAPL,AAPL,AAPL,AAPL
0,2023-01-03,122.982719,128.715417,122.097738,128.105769,112117500
1,2023-01-04,124.251183,126.512801,122.992546,124.772336,89113600
2,2023-01-05,122.933533,125.637638,122.677877,125.008319,80962700
3,2023-01-06,127.456772,128.115588,122.805715,123.907026,87754700
4,2023-01-09,127.977913,131.183516,127.722257,128.292580,70790800


In [10]:
import pandas as pd

if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

df = df.reset_index()

In [11]:
print(type(df.loc[0, "Volume"]))

<class 'numpy.int64'>


In [12]:
#transform
df["return"] = df["Close"].pct_change()

# rolling volatility 20 hari
df["volatility_20d"] = df["return"].rolling(20).std()

df = df.dropna()
df.head()

Price,index,date,Close,High,Low,Open,Volume,return,volatility_20d
20,20,2023-02-01,143.002914,144.163229,138.961521,141.567289,77663600,0.007901,0.012758
21,21,2023-02-02,148.302963,148.656941,145.697180,146.414993,118339000,0.037063,0.014354
22,22,2023-02-03,151.921555,154.753495,145.362872,145.559531,154357300,0.024400,0.013969
23,23,2023-02-06,149.197800,150.544946,148.263657,150.023792,69858300,-0.017929,0.013955
24,24,2023-02-07,152.069061,152.639384,148.125989,148.125989,83322600,0.019245,0.014142


In [13]:
df = df.reset_index(drop=True)

df["Volume"] = df["Volume"].astype(int)
df["return"] = df["return"].astype(float)
df["volatility_20d"] = df["volatility_20d"].astype(float)

In [14]:
conn = psycopg2.connect(
    host="postgres",
    port=5432,
    database="batch_db",
    user="airflow",
    password="airflow"
)
cur = conn.cursor()

In [15]:
df.columns = df.columns.str.lower()

In [16]:
for _, row in df.iterrows():
    cur.execute("""
        INSERT INTO yahoo_finance.stock_price
        (symbol, date, open, high, low, close, volume, return, volatility_20d)
        VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s)
    """, (
        symbol,
        row["date"],
        row["open"],
        row["high"],
        row["low"],
        row["close"],
        int(row["volume"]),
        float(row["return"]),
        float(row["volatility_20d"])
    ))

conn.commit()
cur.close()
conn.close()

print("✅ Historical data loaded to Postgres")

✅ Historical data loaded to Postgres
